# XTTS / Coqui TTS — ноўтбук, перароблены пад `tuteishygpt/coqui-ai-TTS`

Гэты варыянт прывязаны да структуры і API рэпазіторыя `coqui-ai-TTS`:

- выкарыстоўвае `TTS`-модулі і рэцэпт XTTS v2;
- больш **не** залежыць ад `XTTSv2-Finetuning-for-New-Languages`;
- не выкарыстоўвае `standalone_setup.sh`, `train_dvae_xtts.py`, `extend_vocab_config.py` і іншыя скрыпты, якіх няма ў гэтым рэпазіторыі;
- прыбірае захардкоджаныя сакрэты;
- падтрымлівае як `metadata_train.csv` / `metadata_eval.csv`, так і адзін `metadata.csv` з аўтаматычным split;
- пакуе фінальны run у самадастатковую папку з `model.pth`, `config.json`, `vocab.json`, `dvae.pth`, `mel_stats.pth`.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuteishygpt/coqui-ai-TTS.git"
REPO_BRANCH = "dev"
REPO_DIR = Path("coqui-ai-TTS")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())


Current working directory: /content/coqui-ai-TTS


In [2]:
import importlib
import subprocess
import sys

def ensure_package(module_name: str, pip_name: str | None = None):
    pip_name = pip_name or module_name
    try:
        importlib.import_module(module_name)
        print(f"{module_name}: already installed")
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-U", pip_name], check=True)
        print(f"{module_name}: installed")

# Coqui TTS from source does not automatically guarantee torch in every environment,
# so we check it explicitly.
ensure_package("torch")
ensure_package("torchaudio")
ensure_package("kagglehub")
ensure_package("huggingface_hub")
ensure_package("soundfile")

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]"], check=True)

print("Environment is ready.")


torch: already installed
torchaudio: already installed
kagglehub: already installed
huggingface_hub: already installed
soundfile: already installed
Environment is ready.


In [3]:
%%bash
echo "==== OS ===="
uname -a
cat /etc/os-release || true

echo "==== Python ===="
python --version
which python

echo "==== GPU ===="
nvidia-smi || echo "nvidia-smi not available"


==== OS ====
Linux 586d30acf806 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
==== Python ====
Python 3.12.13
/usr/local/bin/python
==== GPU ====
Wed Mar 25 09:35:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/

In [4]:
from pathlib import Path

# Варыянт 1: спампаваць датасэт з KaggleHub
USE_KAGGLEHUB = True
KAGGLE_DATASET = "wisekinder/by-bel-korpus-audio-set-1"

# Варыянт 2: выкарыстоўваць ужо існуючую лакальную папку
LOCAL_DATASET_SOURCE = None  # прыклад: "/kaggle/input/my-dataset"

if USE_KAGGLEHUB:
    import kagglehub
    SOURCE_DATASET_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
else:
    if not LOCAL_DATASET_SOURCE:
        raise ValueError("Set LOCAL_DATASET_SOURCE when USE_KAGGLEHUB=False")
    SOURCE_DATASET_DIR = Path(LOCAL_DATASET_SOURCE).expanduser().resolve()

print("SOURCE_DATASET_DIR =", SOURCE_DATASET_DIR)


100%|██████████| 3.61G/3.61G [04:01<00:00, 16.1MB/s]

Extracting files...


SOURCE_DATASET_DIR = /root/.cache/kagglehub/datasets/wisekinder/by-bel-korpus-audio-set-1/versions/4


In [5]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset-1"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Капіюем, а не move — каб зыходныя файлы не знікалі.
if SOURCE_DATASET_DIR.resolve() != DATASET_DIR.resolve():
    shutil.copytree(SOURCE_DATASET_DIR, DATASET_DIR, dirs_exist_ok=True)

print("DATASET_DIR =", DATASET_DIR)
print("Top-level files:", sorted([p.name for p in DATASET_DIR.iterdir()])[:30])


DATASET_DIR = /content/coqui-ai-TTS/dataset-1
Top-level files: ['metadata_eval.csv', 'metadata_train.csv', 'transcription.csv', 'wavs']


In [6]:
from pathlib import Path
import random

DATASET_DIR = Path.cwd() / "dataset-1"

train_meta = DATASET_DIR / "metadata_train.csv"
eval_meta = DATASET_DIR / "metadata_eval.csv"
single_meta = DATASET_DIR / "metadata.csv"

if train_meta.exists() and eval_meta.exists():
    print("Using existing metadata_train.csv and metadata_eval.csv")
elif single_meta.exists():
    lines = [line.strip() for line in single_meta.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(lines) < 2:
        raise ValueError("metadata.csv contains too few rows to create a train/eval split")

    rnd = random.Random(42)
    rnd.shuffle(lines)

    eval_n = max(1, min(256, int(len(lines) * 0.02)))
    eval_lines = lines[:eval_n]
    train_lines = lines[eval_n:]

    train_meta.write_text("\n".join(train_lines) + "\n", encoding="utf-8")
    eval_meta.write_text("\n".join(eval_lines) + "\n", encoding="utf-8")

    print(f"Created metadata_train.csv ({len(train_lines)} rows)")
    print(f"Created metadata_eval.csv ({len(eval_lines)} rows)")
else:
    raise FileNotFoundError(
        "Expected either metadata_train.csv + metadata_eval.csv or a single metadata.csv inside dataset-1"
    )

print("train_meta =", train_meta)
print("eval_meta  =", eval_meta)

def preview_lines(path: Path, n: int = 3):
    rows = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"\nPreview from {path.name}:")
    for row in rows[:n]:
        print(row)

preview_lines(train_meta)
preview_lines(eval_meta)


Using existing metadata_train.csv and metadata_eval.csv
train_meta = /content/coqui-ai-TTS/dataset-1/metadata_train.csv
eval_meta  = /content/coqui-ai-TTS/dataset-1/metadata_eval.csv

Preview from metadata_train.csv:
audio_file|text|speaker_name
wavs/01776.wav|Надысь мех пшаніцы завезлі... А не нудно там, на чужыне? Аднаму, без маткі?|@speaker1
wavs/07129.wav|што не толькі можа, а што ўжо стаў гаспадаром і ўладаром жаданай гэтай зямлі,|@speaker1

Preview from metadata_eval.csv:
audio_file|text|speaker_name
wavs/02466.wav|паабяцаў! Так і перадай! Перадам! Харчаў павярнуўся і, важка, моцна ступаючы, падаўся ў пакой.|@speaker1
wavs/03643.wav|не наліліся жывой ружовасцю. Ён жа то стагнаў, нібы ў сне, то скрыпеў зубамі і нешта мармытаў без складу, без ладу.|@speaker1


In [7]:
from pathlib import Path

# Калі хочаш працягнуць fine-tuning не з афіцыйнага XTTS-v2, а са свайго HF-рэпа,
# задай BASE_MODEL_REPO_ID.
BASE_MODEL_REPO_ID = "archivartaunik/BE_XTTS_V3_10epoch1DatSmal_8ep_Kir250V2"

CHECKPOINTS_DIR = Path.cwd() / "checkpoints"
BASE_MODEL_DIR = CHECKPOINTS_DIR / "XTTS_base"
BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if BASE_MODEL_REPO_ID:
    from huggingface_hub import snapshot_download

    print(f"Downloading custom base model from {BASE_MODEL_REPO_ID} ...")
    snapshot_download(
        repo_id=BASE_MODEL_REPO_ID,
        local_dir=str(BASE_MODEL_DIR),
        local_dir_use_symlinks=False,
    )
    print("BASE_MODEL_DIR =", BASE_MODEL_DIR)
else:
    print("BASE_MODEL_REPO_ID is not set -> training script will use official coqui/XTTS-v2 files.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

BASE_MODEL_DIR = /content/coqui-ai-TTS/checkpoints/XTTS_base


In [8]:
from pathlib import Path

train_script_path = Path.cwd() / "train_xtts_be.py"
train_script_path.write_text('from __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom trainer import Trainer, TrainerArgs\nfrom TTS.config.shared_configs import BaseDatasetConfig\nfrom TTS.tts.configs.xtts_config import XttsAudioConfig\nfrom TTS.tts.datasets import load_tts_samples\nfrom TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig\nfrom TTS.utils.manage import ModelManager\n\nPROJECT_ROOT = Path(__file__).resolve().parent\nDATASET_DIR = PROJECT_ROOT / "dataset-1"\nTRAIN_META = DATASET_DIR / "metadata_train.csv"\nEVAL_META = DATASET_DIR / "metadata_eval.csv"\n\nRUN_NAME = "GPT_XTTS_BE_FT"\nPROJECT_NAME = "XTTS_trainer"\nDASHBOARD_LOGGER = "tensorboard"\nLOGGER_URI = None\n\nOUT_PATH = PROJECT_ROOT / "run" / "training"\nCHECKPOINTS_OUT_PATH = PROJECT_ROOT / "checkpoints" / "XTTS_base"\nCHECKPOINTS_OUT_PATH.mkdir(parents=True, exist_ok=True)\nOUT_PATH.mkdir(parents=True, exist_ok=True)\n\nLANGUAGE = "be"\nBATCH_SIZE = 10\nGRAD_ACUMM_STEPS = 20\nNUM_EPOCHS = 5\nLEARNING_RATE = 9e-6\nSAVE_STEP = 24000\nSTART_WITH_EVAL = True\nOPTIMIZER_WD_ONLY_ON_WEIGHTS = True\n\nTEST_SENTENCE = (\n    "Гэта тэставае сказанне для праверкі беларускага XTTS пасля fine-tuning."\n)\n\ndef first_existing(*paths: Path) -> Path | None:\n    for path in paths:\n        if path and path.exists():\n            return path\n    return None\n\ndef parse_reference_wav(meta_path: Path, dataset_dir: Path) -> str:\n    rows = [line.strip() for line in meta_path.read_text(encoding="utf-8").splitlines() if line.strip()]\n    if not rows:\n        raise RuntimeError(f"No rows found in {meta_path}")\n    rel_wav = rows[0].split("|")[0].strip()\n    wav_path = dataset_dir / rel_wav\n    if not wav_path.exists():\n        raise FileNotFoundError(f"Reference wav not found: {wav_path}")\n    return str(wav_path)\n\ndef resolve_base_model_files(custom_base_dir: Path | None):\n    dvae_link = "https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth"\n    mel_link = "https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth"\n    vocab_link = "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json"\n    model_link = "https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth"\n\n    dvae_path = CHECKPOINTS_OUT_PATH / "dvae.pth"\n    mel_path = CHECKPOINTS_OUT_PATH / "mel_stats.pth"\n    vocab_path = CHECKPOINTS_OUT_PATH / "vocab.json"\n    model_path = CHECKPOINTS_OUT_PATH / "model.pth"\n\n    if custom_base_dir and custom_base_dir.exists():\n        custom_dvae = first_existing(custom_base_dir / "dvae.pth")\n        custom_mel = first_existing(custom_base_dir / "mel_stats.pth")\n        custom_vocab = first_existing(custom_base_dir / "vocab.json")\n        custom_model = first_existing(\n            custom_base_dir / "model.pth",\n            custom_base_dir / "best_model.pth",\n        )\n\n        if custom_vocab and custom_model:\n            # Copy files into a stable local folder used by training + packaging.\n            import shutil\n\n            shutil.copy2(custom_vocab, vocab_path)\n            shutil.copy2(custom_model, model_path)\n\n            if custom_dvae:\n                shutil.copy2(custom_dvae, dvae_path)\n            if custom_mel:\n                shutil.copy2(custom_mel, mel_path)\n\n    missing = []\n    if not dvae_path.exists():\n        missing.append(dvae_link)\n    if not mel_path.exists():\n        missing.append(mel_link)\n    if not vocab_path.exists():\n        missing.append(vocab_link)\n    if not model_path.exists():\n        missing.append(model_link)\n\n    if missing:\n        print("Downloading missing XTTS base files ...")\n        ModelManager._download_model_files(missing, str(CHECKPOINTS_OUT_PATH), progress_bar=True)\n\n    return str(model_path), str(vocab_path), str(dvae_path), str(mel_path)\n\ndef main():\n    if not TRAIN_META.exists():\n        raise FileNotFoundError(f"Missing train metadata: {TRAIN_META}")\n    if not EVAL_META.exists():\n        raise FileNotFoundError(f"Missing eval metadata: {EVAL_META}")\n\n    custom_base_dir = PROJECT_ROOT / "checkpoints" / "XTTS_base"\n    xtts_checkpoint, tokenizer_file, dvae_checkpoint, mel_norm_file = resolve_base_model_files(custom_base_dir)\n\n    speaker_reference = [parse_reference_wav(EVAL_META if EVAL_META.exists() else TRAIN_META, DATASET_DIR)]\n\n    config_dataset = BaseDatasetConfig(\n        formatter="ljspeech",\n        dataset_name="belarusian_custom",\n        path=str(DATASET_DIR),\n        meta_file_train=TRAIN_META.name,\n        meta_file_val=EVAL_META.name,\n        language=LANGUAGE,\n    )\n\n    datasets = [config_dataset]\n\n    model_args = GPTArgs(\n        max_conditioning_length=132300,\n        min_conditioning_length=66150,\n        debug_loading_failures=False,\n        max_wav_length=255995,\n        max_text_length=200,\n        mel_norm_file=mel_norm_file,\n        dvae_checkpoint=dvae_checkpoint,\n        xtts_checkpoint=xtts_checkpoint,\n        tokenizer_file=tokenizer_file,\n        gpt_num_audio_tokens=1026,\n        gpt_start_audio_token=1024,\n        gpt_stop_audio_token=1025,\n        gpt_use_masking_gt_prompt_approach=True,\n        gpt_use_perceiver_resampler=True,\n    )\n\n    audio_config = XttsAudioConfig(\n        sample_rate=22050,\n        dvae_sample_rate=22050,\n        output_sample_rate=24000,\n    )\n\n    config = GPTTrainerConfig(\n        output_path=str(OUT_PATH),\n        model_args=model_args,\n        run_name=RUN_NAME,\n        project_name=PROJECT_NAME,\n        run_description="Belarusian GPT XTTS training on coqui-ai-TTS",\n        dashboard_logger=DASHBOARD_LOGGER,\n        logger_uri=LOGGER_URI,\n        audio=audio_config,\n        batch_size=BATCH_SIZE,\n        batch_group_size=48,\n        eval_batch_size=max(1, BATCH_SIZE),\n        num_loader_workers=4,\n        eval_split_max_size=256,\n        epochs=NUM_EPOCHS,\n        print_step=50,\n        plot_step=100,\n        log_model_step=1000,\n        save_step=SAVE_STEP,\n        save_n_checkpoints=3,\n        save_checkpoints=True,\n        print_eval=False,\n        optimizer="AdamW",\n        optimizer_wd_only_on_weights=OPTIMIZER_WD_ONLY_ON_WEIGHTS,\n        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},\n        lr=LEARNING_RATE,\n        lr_scheduler="MultiStepLR",\n        lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},\n        test_sentences=[\n            {\n                "text": TEST_SENTENCE,\n                "speaker_wav": speaker_reference,\n                "language": LANGUAGE,\n            }\n        ],\n        languages=[LANGUAGE],\n    )\n\n    model = GPTTrainer.init_from_config(config)\n\n    train_samples, eval_samples = load_tts_samples(\n        datasets,\n        eval_split=True,\n        eval_split_max_size=config.eval_split_max_size,\n        eval_split_size=config.eval_split_size,\n    )\n\n    trainer = Trainer(\n        TrainerArgs(\n            restore_path=None,\n            skip_train_epoch=False,\n            start_with_eval=START_WITH_EVAL,\n            grad_accum_steps=GRAD_ACUMM_STEPS,\n        ),\n        config,\n        output_path=str(OUT_PATH),\n        model=model,\n        train_samples=train_samples,\n        eval_samples=eval_samples,\n    )\n    trainer.fit()\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print("Written:", train_script_path)
print(train_script_path.read_text(encoding="utf-8")[:2000])


Written: /content/coqui-ai-TTS/train_xtts_be.py
from __future__ import annotations

import os
from pathlib import Path

from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.utils.manage import ModelManager

PROJECT_ROOT = Path(__file__).resolve().parent
DATASET_DIR = PROJECT_ROOT / "dataset-1"
TRAIN_META = DATASET_DIR / "metadata_train.csv"
EVAL_META = DATASET_DIR / "metadata_eval.csv"

RUN_NAME = "GPT_XTTS_BE_FT"
PROJECT_NAME = "XTTS_trainer"
DASHBOARD_LOGGER = "tensorboard"
LOGGER_URI = None

OUT_PATH = PROJECT_ROOT / "run" / "training"
CHECKPOINTS_OUT_PATH = PROJECT_ROOT / "checkpoints" / "XTTS_base"
CHECKPOINTS_OUT_PATH.mkdir(parents=True, exist_ok=True)
OUT_PATH.mkdir(parents=True, exist_ok=True)

LANGUAGE = "be"
BATCH_SIZE = 10
GRAD_ACUMM_

In [9]:
# Запуск fine-tuning.
# Пры патрэбе змяні CUDA_VISIBLE_DEVICES на свой нумар GPU.

import os
import subprocess
import sys

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = env.get("CUDA_VISIBLE_DEVICES", "0")

subprocess.run([sys.executable, "train_xtts_be.py"], check=True, env=env)


CalledProcessError: Command '['/usr/bin/python3', 'train_xtts_be.py']' returned non-zero exit status 1.

In [11]:
!python -u train_xtts_be_fixed_modern_torch_no_vocab_patch_bf16fix.py

TF32 not enabled: current GPU / PyTorch build does not support TF32.
cuDNN benchmark enabled.
enable_flash_sdp(True)
enable_mem_efficient_sdp(True)
enable_math_sdp(True)
AMP requested: True
AMP precision (trainer): bf16
XTTS_AMP_PRECISION raw env: None
Torch compile requested: False
Normalized metadata: metadata_train.csv -> metadata_train.ljspeech.csv (8152 rows)
Normalized metadata: metadata_eval.csv -> metadata_eval.ljspeech.csv (430 rows)
Applied trainer speed config:
  config.mixed_precision = True
  config.precision = bf16
  config.num_loader_workers = 2
torch.compile disabled. Set XTTS_USE_TORCH_COMPILE=1 to enable it.
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: bf16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: True
 | > Torch TF32 MatMul: False
2026-03-25 10:26:26.961914: E exter

In [17]:
!python -u train_xtts_be_fixed_modern_torch_no_vocab_patch_bf16fix_cli.py \
  --output_path checkpoints/ \
  --tf32_matmul --tf32_cudnn \
  --batch_size 6 --grad_acumm 12 \
  --save_step 2400 \
  --lr 9e-6 \
  --use_fp16 \
  --num_epochs 1

Launch config:
  output_path = /content/coqui-ai-TTS/checkpoints
  batch_size = 6
  grad_accum_steps = 12
  save_step = 2400
  lr = 9e-06
  num_epochs = 1
  use_augmentation = False
  monitor_gradients = False
  compile_model = False
  tf32_matmul = True
  tf32_cudnn = True
  use_amp = True
  amp_precision = fp16
  num_loader_workers = 2
TF32 not enabled: current GPU / PyTorch build does not support TF32.
cuDNN benchmark enabled.
enable_flash_sdp(True)
enable_mem_efficient_sdp(True)
enable_math_sdp(True)
AMP requested: True
AMP precision (trainer): fp16
Torch compile requested: False
Normalized metadata: metadata_train.csv -> metadata_train.ljspeech.csv (8152 rows)
Normalized metadata: metadata_eval.csv -> metadata_eval.ljspeech.csv (430 rows)
Applied trainer speed config:
  config.mixed_precision = True
  config.precision = fp16
  config.num_loader_workers = 2
  config.model_param_stats = False
torch.compile disabled. Use --compile_model to enable it.
 > Training Environment:
 | > Bac

In [ ]:
!python -u train_xtts_be_fixed_modern_torch_no_vocab_patch_bf16fix_cli.py \
  --output_path checkpoints/ \
  --batch_size 4 \
  --grad_acumm 12 \
  --save_step 2400 \
  --lr 9e-6 \
  --num_epochs 1 \
  --no_mixed_precision

Launch config:
  output_path = /content/coqui-ai-TTS/checkpoints
  batch_size = 4
  grad_accum_steps = 12
  save_step = 2400
  lr = 9e-06
  num_epochs = 1
  use_augmentation = False
  monitor_gradients = False
  compile_model = False
  tf32_matmul = True
  tf32_cudnn = True
  use_amp = False
  amp_precision = bf16
  num_loader_workers = 2
TF32 not enabled: current GPU / PyTorch build does not support TF32.
cuDNN benchmark enabled.
enable_flash_sdp(True)
enable_mem_efficient_sdp(True)
enable_math_sdp(True)
AMP requested: False
AMP precision (trainer): bf16
Torch compile requested: False
Normalized metadata: metadata_train.csv -> metadata_train.ljspeech.csv (8152 rows)
Normalized metadata: metadata_eval.csv -> metadata_eval.ljspeech.csv (430 rows)
Applied trainer speed config:
  config.mixed_precision = False
  config.precision = float32
  config.num_loader_workers = 2
  config.model_param_stats = False
torch.compile disabled. Use --compile_model to enable it.
 > Training Environment:
 |

In [ ]:
from pathlib import Path
p = Path("/content/coqui-ai-TTS/dataset-1/metadata_train.csv")
print(p.read_text(encoding="utf-8").splitlines()[:5])

In [ ]:
from pathlib import Path

dataset_dir = Path("/content/coqui-ai-TTS/dataset-1")
meta_path = dataset_dir / "metadata_train.csv"
wav_dir = dataset_dir / "wavs"

meta_lines = [x.strip() for x in meta_path.read_text(encoding="utf-8").splitlines() if x.strip()]
wav_names = {p.name for p in wav_dir.glob("*.wav")}

rows = []
for i, line in enumerate(meta_lines):
    cols = [c.strip() for c in line.split("|")]
    if not cols:
        continue
    if i == 0 and cols[0].lower() == "audio_file":
        continue
    rel = Path(cols[0]).name
    rows.append(rel)

missing = [x for x in rows if x not in wav_names]

print("Metadata rows:", len(rows))
print("Wav files:", len(wav_names))
print("Missing from wav folder:", len(missing))
print("First 20 missing:", missing[:20])

In [ ]:
from pathlib import Path
import shutil

RUNS_DIR = Path.cwd() / "run" / "training"
run_dirs = [p for p in RUNS_DIR.iterdir() if p.is_dir()]
if not run_dirs:
    raise RuntimeError(f"No run directories found in {RUNS_DIR}")

LATEST_RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)
print("LATEST_RUN_DIR =", LATEST_RUN_DIR)

def pick_checkpoint(run_dir: Path) -> Path:
    candidates = [
        run_dir / "model.pth",
        run_dir / "best_model.pth",
    ]
    checkpoint_files = sorted(run_dir.glob("checkpoint_*.pth"))
    if checkpoint_files:
        candidates.extend(checkpoint_files[::-1])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No checkpoint found in {run_dir}")

best_checkpoint = pick_checkpoint(LATEST_RUN_DIR)
packaged_model = LATEST_RUN_DIR / "model.pth"
if best_checkpoint != packaged_model:
    shutil.copy2(best_checkpoint, packaged_model)
    print(f"Copied {best_checkpoint.name} -> model.pth")
else:
    print("model.pth already exists")

ASSET_SOURCE = Path.cwd() / "checkpoints" / "XTTS_base"
for asset_name in ("vocab.json", "dvae.pth", "mel_stats.pth"):
    src = ASSET_SOURCE / asset_name
    dst = LATEST_RUN_DIR / asset_name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"Copied {asset_name} into run dir")

print("Packaged run contents:")
for p in sorted(LATEST_RUN_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import re
import torch
import torchaudio
from IPython.display import Audio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Corrected: Use Path object and the actual latest run directory from the finetuning output
LATEST_RUN_DIR = Path("/content/coqui-ai-TTS/run/training/GPT_XTTS_BE_FT-March-24-2026_05+34PM-61cdea63")
device = "cuda" if torch.cuda.is_available() else "cpu"


CONFIG_PATH = LATEST_RUN_DIR / "config.json"
MODEL_PATH = LATEST_RUN_DIR / "model.pth"
VOCAB_PATH = LATEST_RUN_DIR / "vocab.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)
if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)
if not VOCAB_PATH.exists():
    raise FileNotFoundError(VOCAB_PATH)

# Задай тут свой рэферэнсны аўдыёфайл.
SPEAKER_AUDIO_FILE = "/content/coqui-ai-TTS/dataset-1/wavs/00014.wav"

if SPEAKER_AUDIO_FILE is None:
    # Бярэм першы wav з eval metadata як fallback.
    first_eval_row = [line.strip() for line in (Path.cwd() / "dataset-1" / "metadata_eval.csv").read_text(encoding="utf-8").splitlines() if line.strip()][0]
    SPEAKER_AUDIO_FILE = str((Path.cwd() / "dataset-1" / first_eval_row.split("|")[0]).resolve())

tts_text = '''
«Гэтая крама — мой адзіны бізнэс». Што гавораць работнікі гандлёвага цэнтра, які на тым тыдні згарэў у Віцебску

У найбліжэйшы час арандатары змогуць трапіць усярэдзіну ГЦ, каб забраць адтуль ацалелыя тавары. Калі, вядома, будзе што забіраць.
'''.strip()

lang = "be"

config = XttsConfig()
config.load_json(str(CONFIG_PATH))

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path=str(MODEL_PATH),
    vocab_path=str(VOCAB_PATH),
    use_deepspeed=False,
)
model.to(device)

gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
    audio_path=[SPEAKER_AUDIO_FILE]
)

sentences = [s.strip() for s in re.split(r"(?<=[.!?…])\s+|\n+", tts_text) if s.strip()]

wav_chunks = []
for sentence in sentences:
    out = model.inference(
        text=sentence,
        language=lang,
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.2,
        length_penalty=1.0,
        repetition_penalty=10.0,
        top_k=20,
        top_p=0.6,
    )
    wav_chunks.append(torch.tensor(out["wav"]))

out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0).cpu()
output_wav_path = LATEST_RUN_DIR / "inference_be.wav"
torchaudio.save(str(output_wav_path), out_wav, 24000)

print("Saved:", output_wav_path)
Audio(out_wav.numpy(), rate=24000)

In [ ]:
import platform
import sys
import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA (compiled):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuDNN version:", torch.backends.cudnn.version())
    print("cuDNN enabled:", torch.backends.cudnn.enabled)
    print("Device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU{i}: {p.name}, CC {p.major}.{p.minor}, VRAM {round(p.total_memory / 1024**3, 2)} GB")
